# Actividad 6: Explorando transformers preentrenados con Hugging Face

**Curso:** Deep Learning  
**Profesor:** Gonzalo A. Ruz  
**Ayudante:** Anthony D. Cho

En esta actividad aplicaremos modelos transformer preentrenados a distintas tareas de procesamiento de lenguaje natural usando `pipeline()`.

Trabajaremos con ejemplos simples y guiados para observar cómo cambian las capacidades del modelo según la tarea.

## Objetivos de la actividad

Al finalizar esta actividad deberías poder:

- usar `pipeline()` para distintas tareas de NLP,
- interpretar salidas de modelos preentrenados,
- comparar distintos usos de transformers,
- reflexionar sobre la diferencia entre modelos clásicos y modelos modernos preentrenados.

## Instrucciones
- La actividad debe ser realizada por los grupos de trabajo
- Responda cada pregunta en las celdas correspondientes
- Justifique brevemente sus respuestas cuando se solicite
- Renombrar el archivo agregando el apellido de las y los integrantes, por ejemplo Actividad6_Tupper_Tudor_Gorosito_Acosta.ipynb
- Subir el archivo al link de entrega Actividad 6 en webcursos que será habilitado
- __Fecha de entrega:__ Idealmente al final del bloque 2 de la clase del 27 de abril 2026. Fecha límite de entrega 4 de mayo 2026

## Integrantes (RUT - Nombre y apellido):

- 13.257.556-8 - Ricardo Lopez
- 16.789.149-7 - Camilo Muñoz
- 13.307.082-6 - Álvaro Iriarte
- 25.608.509-7 - Ranse Vidal


## Instrucciones generales

- Completa las celdas indicadas.
- Lee con atención los comentarios y preguntas.
- Responde las preguntas breves en las celdas Markdown donde se indique.
- Si alguna salida cambia levemente, no te preocupes: eso puede ocurrir según la versión del modelo o del entorno.

In [56]:
!pip -q install -U transformers accelerate sentencepiece

import random
import numpy as np
import torch
from transformers import pipeline

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 0 if torch.cuda.is_available() else -1

print(f"Dispositivo: {'GPU' if device == 0 else 'CPU'}")
print(f"Semilla: {SEED}")

Dispositivo: GPU
Semilla: 42


```markdown
# ⚠️ Si Colab pide reiniciar, reiniciar y ejecutar TODO nuevamente
```

In [57]:
# La configuración del dispositivo y la semilla ya se realizaron en la celda anterior.
# Esta celda se mantiene para no alterar la estructura, pero su funcionalidad es redundante ahora.
pass

> **Importante:** si Colab pide reiniciar el entorno, hazlo antes de continuar.

## Parte 1: análisis de sentimiento

Comenzaremos con una tarea clásica de NLP: clasificar si un texto expresa un sentimiento positivo o negativo.

Completa la siguiente celda para crear un pipeline de `sentiment-analysis`.

In [58]:
from transformers import pipeline

sentiment_pipeline = pipeline("sentiment-analysis", device=device)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Ahora aplica el modelo a tres textos de ejemplo y observa las predicciones.

In [59]:
texts = [
    "I loved this product. It is easy to use and works very well.",
    "This was a disappointing purchase. The quality is poor.",
    "The product is acceptable, but I expected something better."
]

In [60]:
sentiment_results = sentiment_pipeline(texts)

positive_texts = [
    text for text, res in zip(texts, sentiment_results)
    if res["label"] == "POSITIVE"
]

most_uncertain = min(
    zip(texts, sentiment_results),
    key=lambda x: abs(x[1]["score"] - 0.5)
)

most_uncertain_text = most_uncertain[0]
most_uncertain_score = most_uncertain[1]["score"]

print("Resultados de sentimiento:")
for text, result in zip(texts, sentiment_results):
    print(f"- Texto: {text}")
    print(f"  Predicción: {result['label']} | Score: {result['score']:.4f}")

print("\nTextos positivos:")
for t in positive_texts:
    print(f"- {t}")

print("\nTexto con mayor incertidumbre:")
print(f"- {most_uncertain_text}")
print(f"- Score asociado: {most_uncertain_score:.4f}")

Resultados de sentimiento:
- Texto: I loved this product. It is easy to use and works very well.
  Predicción: POSITIVE | Score: 0.9999
- Texto: This was a disappointing purchase. The quality is poor.
  Predicción: NEGATIVE | Score: 0.9998
- Texto: The product is acceptable, but I expected something better.
  Predicción: NEGATIVE | Score: 0.9093

Textos positivos:
- I loved this product. It is easy to use and works very well.

Texto con mayor incertidumbre:
- The product is acceptable, but I expected something better.
- Score asociado: 0.9093


### Pregunta 1

Observa las predicciones del modelo.

**Escribe una breve respuesta:**
- ¿Qué textos fueron clasificados como positivos?
- ¿Cuál parece generar más incertidumbre según el score?

**Respuesta:**

- Los textos clasificados como positivos son: `{', '.join(positive_texts)}`.
- El texto que generó mayor incertidumbre fue: `"{most_uncertain_text}"` con un score de `{most_uncertain_score:.4f}`, lo que indica que su predicción estuvo más cerca de 0.5.

## Parte 2: zero-shot classification

Ahora usaremos un pipeline que permite clasificar un texto con etiquetas propuestas por nosotros, sin entrenar un clasificador específico para esas clases.

Completa la siguiente celda con el pipeline adecuado.

In [61]:
zero_shot_pipeline = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device
)

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [62]:
candidate_labels = ["business", "health", "travel", "food"]

text = "This article reviews the nutritional benefits of eating vegetables, fruits, and whole grains every day."

zero_shot_result = zero_shot_pipeline(text, candidate_labels)
best_label = zero_shot_result["labels"][0]
best_score = zero_shot_result["scores"][0]

print("Ranking zero-shot:")
for label, score in zip(zero_shot_result["labels"], zero_shot_result["scores"]):
    print(f"- {label}: {score:.4f}")

print(f"Mejor etiqueta: {best_label} ({best_score:.4f})")

Ranking zero-shot:
- food: 0.5120
- health: 0.4817
- business: 0.0037
- travel: 0.0027
Mejor etiqueta: food (0.5120)


### Pregunta 2

Según la salida del modelo:

- ¿cuál fue la etiqueta más probable?
- ¿te parece razonable esa predicción?
- ¿qué ventaja tiene este enfoque frente a entrenar un clasificador desde cero para cada conjunto de etiquetas?

**Respuesta:**

- La etiqueta más probable fue `{best_label}` con un puntaje de `{best_score:.4f}`.
- Sí, es razonable porque el texto trata explícitamente de beneficios nutricionales y alimentación saludable.
- La ventaja clave de este enfoque es su flexibilidad: permite clasificar textos con etiquetas personalizadas sin necesidad de entrenar un clasificador desde cero para cada conjunto de clases, aprovechando el conocimiento preentrenado del modelo.

## Parte 3: question answering

Ahora probaremos una tarea de pregunta-respuesta.

En este caso, el modelo recibe:

- una **pregunta**,
- un **contexto**,

e intenta encontrar la mejor respuesta dentro del texto entregado.

Completa la siguiente celda con el pipeline adecuado.

In [63]:
qa_pipeline = pipeline("question-answering", device=device)

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [64]:
context = """
The Amazon rainforest is the largest tropical rainforest in the world.
It covers parts of several South American countries, with most of it located in Brazil.
The rainforest is known for its extraordinary biodiversity and plays an important role in regulating the global climate.
"""

question = "Which country contains most of the Amazon rainforest?"

qa_result = qa_pipeline(question=question, context=context)
qa_answer = qa_result["answer"]
qa_score = qa_result["score"]

print(f"Respuesta QA: {qa_answer}")
print(f"Score QA: {qa_score:.4f}")

Respuesta QA: Brazil
Score QA: 0.9891


### Pregunta 3

Mira la salida del modelo.

- ¿Qué respuesta encontró?
- ¿Por qué esta tarea es distinta de `text-generation`?

**Respuesta:**

- El modelo encontró `"{qa_answer}"` como respuesta, con una confianza de `{qa_score:.4f}`.
- Esta tarea es distinta de `text-generation` porque QA **extrae** un segmento (span) existente del contexto entregado que mejor responde a la pregunta, mientras que `text-generation` **produce** texto nuevo token a token, sin estar restringido a la información explícita del contexto.

## Parte 4: prueba tus propios ejemplos

Ahora escribe:

- un texto propio para análisis de sentimiento,
- un texto propio para zero-shot classification,
- una pregunta propia para el contexto dado.

La idea es experimentar un poco con los modelos.

In [65]:
# Escribe aquí un texto propio
my_text_sentiment = "The workshop was demanding but very rewarding, and I learned a lot."

# Ejecuta el modelo
custom_sentiment_result = sentiment_pipeline(my_text_sentiment)
print(custom_sentiment_result)


[{'label': 'POSITIVE', 'score': 0.9996849298477173}]


In [66]:
# Escribe aquí tu texto y tus etiquetas
my_text_zero_shot = "The company announced a new electric bus fleet to reduce city emissions."
my_candidate_labels = ["technology", "environment", "sports", "politics"]

# Definir variables personalizadas para trazabilidad
custom_zero_shot_text = my_text_zero_shot
custom_candidate_labels = my_candidate_labels

# Ejecuta el modelo
custom_zero_shot_result = zero_shot_pipeline(custom_zero_shot_text, custom_candidate_labels)
custom_zero_shot_best_label = custom_zero_shot_result["labels"][0]
custom_zero_shot_best_score = custom_zero_shot_result["scores"][0]

print(f"Texto: {custom_zero_shot_text}")
print("Ranking zero-shot:")
for label, score in zip(custom_zero_shot_result["labels"], custom_zero_shot_result["scores"]):
    print(f"- {label}: {score:.4f}")
print(f"Mejor etiqueta: {custom_zero_shot_best_label} ({custom_zero_shot_best_score:.4f})")

Texto: The company announced a new electric bus fleet to reduce city emissions.
Ranking zero-shot:
- technology: 0.6277
- environment: 0.3606
- sports: 0.0072
- politics: 0.0046
Mejor etiqueta: technology (0.6277)


In [67]:
context = """
The Amazon rainforest is the largest tropical rainforest in the world.
It covers parts of several South American countries, with most of it located in Brazil.
The rainforest is known for its extraordinary biodiversity and plays an important role in regulating the global climate.
"""

my_question = "What is one global role of the Amazon rainforest?"

# Definir variable personalizada para trazabilidad
custom_question = my_question

# Ejecuta el modelo
custom_qa_result = qa_pipeline(question=custom_question, context=context)
custom_qa_answer = custom_qa_result["answer"]
custom_qa_score = custom_qa_result["score"]

print(f"Pregunta: {custom_question}")
print(f"Respuesta QA: {custom_qa_answer}")
print(f"Score QA: {custom_qa_score:.4f}")

Pregunta: What is one global role of the Amazon rainforest?
Respuesta QA: regulating the global climate
Score QA: 0.8611


### Pregunta 4

Después de probar tus propios ejemplos, responde brevemente:

- ¿qué tarea te pareció más intuitiva?
- ¿cuál te pareció más sorprendente?
- ¿en cuál confiarías más para una aplicación real?

**Respuesta:**

- La tarea más intuitiva fue la clasificación de sentimiento, porque la salida (etiqueta: `{custom_sentiment_label}`, score: `{custom_sentiment_score:.4f}`) es directa y fácil de interpretar.
- La más sorprendente fue Question Answering, ya que identifica una respuesta precisa (`"{custom_qa_answer}"`, score: `{custom_qa_score:.4f}`) dentro de un contexto dado, demostrando una comprensión contextual profunda.
- Para una aplicación real, confiaría más en Question Answering con un contexto controlado, debido a su comportamiento más acotado y verificable. La clasificación zero-shot (`{custom_zero_shot_best_label}`, score: `{custom_zero_shot_best_score:.4f}`) también ofrece flexibilidad, pero su precisión puede variar más con la elección de las etiquetas candidatas.

## Parte 5: generación de texto

Prueba una tarea de generación de texto.

Completa la siguiente celda con el pipeline adecuado.

In [68]:
text_generation_pipeline = pipeline("text-generation", model="distilgpt2", device=device)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Use su propio texto para generar texto nuevo

In [71]:
prompt = "Transformers are useful in education because"

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

generated_result = text_generation_pipeline(
    prompt,
    max_new_tokens=40,
    do_sample=True,
    temperature=0.7,
    pad_token_id=text_generation_pipeline.tokenizer.eos_token_id
)
generated_output = generated_result[0]["generated_text"]

print("Texto generado:")
print(generated_output)

Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Texto generado:
Transformers are useful in education because they can be used to further enhance our learning.
































### Pregunta 5

¿Qué diferencia observas entre:

- clasificar un texto,
- responder una pregunta sobre un contexto,
- generar texto nuevo?

**Respuesta:**

- **Clasificar un texto** (`sentiment-analysis`, `zero-shot-classification`): El modelo asigna una o más etiquetas predefinidas (o candidatas) a un texto completo, categorizándolo.
- **Responder una pregunta sobre un contexto** (`question-answering`): El modelo busca y extrae la respuesta más relevante directamente de un texto dado, basándose en la pregunta.
- **Generar texto nuevo** (`text-generation`): El modelo crea texto original y coherente a partir de un `prompt` inicial, exhibiendo mayor creatividad pero también una mayor variabilidad en los resultados.

## Reflexión final

En la sesión 4 trabajamos con modelos secuenciales como LSTM.
En esta sesión trabajamos con transformers preentrenados.

### Pregunta 6

Escribe una reflexión breve comparando ambos enfoques.

Puedes apoyarte en ideas como:

- entrenamiento desde cero vs modelo preentrenado,
- recurrencia vs atención,
- flexibilidad de tareas,
- facilidad de uso con herramientas modernas.

**Respuesta:**

La principal diferencia entre los modelos secuenciales clásicos como LSTM y los transformers preentrenados radica en su arquitectura y enfoque de procesamiento. Los LSTM procesan el texto de forma **recurrente**, manteniendo un estado oculto para capturar dependencias secuenciales, lo que los hace efectivos para secuencias pero limitados en la captura de relaciones a largo plazo. Por lo general, requieren **entrenamiento desde cero** para cada tarea específica.

Por otro lado, los **Transformers** utilizan el mecanismo de **auto-atención (self-attention)**, que les permite considerar todo el contexto de una secuencia simultáneamente, capturando dependencias globales de manera más eficiente. El gran avance es el **preentrenamiento masivo** en grandes corpus de texto, lo que les dota de un vasto conocimiento lingüístico general. Esto facilita el **transfer learning**, permitiendo que con un ajuste fino mínimo (o incluso sin él, como en `zero-shot-classification`) se adapten a una **flexibilidad de tareas** mucho mayor que los LSTM. Además, herramientas como `pipeline()` de Hugging Face demuestran una **facilidad de uso** superior, encapsulando la complejidad y haciendo que estos modelos sean accesibles para una amplia gama de aplicaciones con un rendimiento sobresaliente, algo impensable con los modelos clásicos para tareas diversas.

In [70]:
required_vars = [
    "SEED",
    "device",
    "sentiment_results",
    "positive_texts",
    "most_uncertain_text",
    "most_uncertain_score",
    "zero_shot_result",
    "best_label",
    "best_score",
    "qa_result",
    "qa_answer",
    "qa_score",
    "custom_sentiment_result",
    "custom_sentiment_label",
    "custom_sentiment_score",
    "custom_zero_shot_result",
    "custom_zero_shot_best_label",
    "custom_zero_shot_best_score",
    "custom_qa_result",
    "custom_qa_answer",
    "custom_qa_score",
    "generated_result",
    "generated_output"
]

missing = [var for var in required_vars if var not in globals()]

if len(missing) == 0:
    print("✅ Validación completa: notebook reproducible y correcto")
else:
    print("❌ Faltan variables:", missing)

✅ Validación completa: notebook reproducible y correcto
